In [19]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

df = pd.read_parquet ('../data/ML_training_data/model_training_data_raw.parquet')

In [20]:
print(df.columns)

Index(['loan_amnt', 'term', 'int_rate', 'installment', 'sub_grade', 'purpose',
       'addr_state', 'annual_inc', 'home_ownership', 'emp_length',
       'verification_status', 'effective_dti', 'is_joint', 'fico_score',
       'revol_util', 'revol_bal', 'open_acc', 'total_acc', 'inq_last_6mths',
       'delinq_2yrs', 'pub_rec', 'pub_rec_bankruptcies', 'mort_acc',
       'earliest_cr_line', 'issue_d', 'loan_status'],
      dtype='str')


In [21]:
print(df['loan_status'].value_counts())
print("- -")
print(df['term'].value_counts())
print("- -")
print(df['emp_length'].value_counts())


loan_status
Fully Paid                                             5197
Current                                                3418
Charged Off                                            1231
Late (31-120 days)                                       69
Does not meet the credit policy. Status:Fully Paid       28
In Grace Period                                          26
Late (16-30 days)                                        19
Does not meet the credit policy. Status:Charged Off      12
Name: count, dtype: int64
- -
term
36 months    7200
60 months    2800
Name: count, dtype: int64
- -
emp_length
10+ years    3217
2 years       866
3 years       816
< 1 year      790
1 year        682
5 years       640
4 years       609
6 years       514
7 years       433
8 years       425
9 years       385
Name: count, dtype: int64


In [22]:
#converting target variable to numerical format

target_map = {
    'Fully Paid': 0,
    'Does not meet the credit policy. Status:Fully Paid': 0,
    'Charged Off': 1,
    'Late (31-120 days)': 1,
    'Does not meet the credit policy. Status:Charged Off': 1,
}

df_clean = df[df['loan_status'].isin(target_map.keys())].copy()

df_clean['target'] = df_clean['loan_status'].map(target_map)

df_clean.drop(columns=['loan_status'], inplace=True)


In [23]:
# parsing time string to numerical data

time_map = {
    ' 36 months': 36,
    ' 60 months': 60,
    '10+ years': 10,
    '< 1 year': 0,
    '1 year': 1,
    '2 years': 2,
    '3 years': 3,
    '4 years': 4,
    '5 years': 5,
    '6 years': 6,
    '7 years': 7,
    '8 years': 8,
    '9 years': 9, 
}

df_clean['emp_year_num'] = df_clean['emp_length'].map(time_map)
df_clean['term_num'] = df_clean['term'].map(time_map)
df_clean.drop(columns=['emp_length', 'term'], inplace=True)


In [24]:
# date engineering
issue_dt = pd.to_datetime(df_clean['issue_d'], format='%b-%Y')
earliest_dt = pd.to_datetime(df_clean['earliest_cr_line'], format='%b-%Y')

df_clean['credit_hist_yrs'] = (issue_dt - earliest_dt).dt.days / 365.25

df_clean['credit_hist_yrs'] = df_clean['credit_hist_yrs'].clip(lower=0)

df_clean.drop(columns=['earliest_cr_line', 'issue_d'], inplace=True)

In [25]:
# Financial ratio engineering

#prep
cols_to_cast = ['installment', 'annual_inc', 'revol_bal', 'open_acc', 'total_acc', 'loan_amnt']
for cols in cols_to_cast:
    df_clean[cols] = pd.to_numeric(df_clean[cols], errors='coerce').astype('float64')

# Monthly loan payment divided by monthly income.
df_clean['installment_to_inc'] = df_clean['installment'] / (df_clean['annual_inc'].replace(0, np.nan) / 12.0)

# Revolving credit balance divided by total annual income.
df_clean['revol_bal_to_inc'] = df_clean['revol_bal'] / df_clean['annual_inc'].replace(0, np.nan)

#Open accounts divided by total accounts.
df_clean['open_acc_ratio'] = df_clean['open_acc'] / df_clean['total_acc'].replace(0, np.nan)

#Total loan amount requested divided by annual income.
df_clean['loan_to_inc'] = df_clean['loan_amnt'] / df_clean['annual_inc'].replace(0, np.nan)

In [26]:
output_path = Path('../data/ML_training_data/model_training_cleaned.parquet')

if not os.path.exists(output_path):
    df_clean.to_parquet(output_path, index=False)